In [2]:
#dataset generation
import pandas as pd
import numpy as np
from faker import Faker
from datetime import datetime, timedelta
import random
from tqdm import tqdm

fake = Faker()
random.seed(42)

# -------------------
# PARAMETERS
# -------------------
NUM_STORES = 10
NUM_PRODUCTS = 500
NUM_CUSTOMERS = 5000
DAYS = 180
TX_PER_DAY = 1000

# -------------------
# STORES
# -------------------
stores = [{
    "store_id": i,
    "store_name": f"Store_{i}",
    "store_city": fake.city(),
    "store_region": fake.state(),
    "opening_date": fake.date_between("-5y", "-1y")
} for i in range(1, NUM_STORES+1)]

stores_df = pd.DataFrame(stores)

# -------------------
# PRODUCTS
# -------------------
categories = ["Electronics","Apparel","Grocery","Home","Books"]

products = [{
    "product_id": i,
    "product_name": fake.word().title(),
    "product_category": random.choice(categories),
    "unit_price": round(random.uniform(50, 5000),2),
    "current_stock_level": random.randint(100,5000)
} for i in range(1, NUM_PRODUCTS+1)]

products_df = pd.DataFrame(products)

# -------------------
# CUSTOMERS
# -------------------
customers = [{
    "customer_id": i,
    "first_name": fake.first_name(),
    "email": fake.email(),
    "loyalty_status": random.choice(["Bronze","Silver","Gold"]),
    "total_loyalty_points": 0,
    "last_purchase_date": None
} for i in range(1, NUM_CUSTOMERS+1)]

customers_df = pd.DataFrame(customers)

# -------------------
# PROMOTIONS
# -------------------
promotions = [{
    "promotion_id": i,
    "promotion_name": f"Promo_{i}",
    "start_date": fake.date_between("-6M","-1M"),
    "end_date": fake.date_between("-1M","+1M"),
    "discount_percentage": round(random.uniform(5,30),2),
    "applicable_category": random.choice(categories + ["ALL"])
} for i in range(1,31)]

promotions_df = pd.DataFrame(promotions)

# -------------------
# LOYALTY RULES
# -------------------
loyalty_rules_df = pd.DataFrame([
    [1,"Standard",0.01,0,0],
    [2,"Big Spender",0.015,2000,100],
    [3,"Premium",0.02,5000,300]
], columns=["rule_id","rule_name","points_per_unit_spend","min_spend_threshold","bonus_points"])

# -------------------
# TRANSACTIONS + LINE ITEMS
# -------------------
headers = []
items = []

tx_id = 1
start_date = datetime.now() - timedelta(days=DAYS)

for d in tqdm(range(DAYS)):
    day = start_date + timedelta(days=d)

    for _ in range(TX_PER_DAY):
        cust = random.randint(1, NUM_CUSTOMERS)
        store = random.randint(1, NUM_STORES)

        total = 0
        item_count = random.randint(1,5)

        for _ in range(item_count):
            product = products_df.sample(1).iloc[0]
            qty = random.randint(1,4)
            amount = qty * product.unit_price

            promo = random.choice(promotions_df.promotion_id.tolist() + [None]*3)

            items.append([
                len(items)+1,
                f"TX{tx_id}",
                product.product_id,
                promo,
                qty,
                round(amount,2)
            ])

            total += amount

        headers.append([
            f"TX{tx_id}",
            cust,
            store,
            day,
            round(total,2),
            fake.msisdn()
        ])

        tx_id += 1

headers_df = pd.DataFrame(headers, columns=[
    "transaction_id","customer_id","store_id",
    "transaction_date","total_amount","customer_phone"
])

items_df = pd.DataFrame(items, columns=[
    "line_item_id","transaction_id","product_id",
    "promotion_id","quantity","line_item_amount"
])

# -------------------
# SAVE ALL 7 DATASETS
# -------------------
stores_df.to_csv("stores.csv", index=False)
products_df.to_csv("products.csv", index=False)
customers_df.to_csv("customer_details.csv", index=False)
promotions_df.to_csv("promotion_details.csv", index=False)
loyalty_rules_df.to_csv("loyalty_rules.csv", index=False)
headers_df.to_csv("store_sales_header.csv", index=False)
items_df.to_csv("store_sales_line_items.csv", index=False)

print("✅ All 7 datasets generated for 6 months!")


100%|██████████| 180/180 [09:02<00:00,  3.01s/it]


✅ All 7 datasets generated for 6 months!


In [37]:
products = pd.read_csv("products.csv", encoding="latin1")
customers = pd.read_csv("customer_details.csv")
loyalty_rules = pd.read_csv("loyalty_rules.csv")
promotions = pd.read_csv("promotions.csv")
sales_header = pd.read_csv("sales_header.csv")
sales_line_items = pd.read_csv("sales_line_items.csv")
stores = pd.read_csv("store.csv", encoding="latin1")

In [17]:
products.head(10)

,product_id,product_name,product_category,unit_price,current_stock_level
0,1,Adidas Running Shoes,Apparel,855,462
1,2,Puma Sports T-Shirt,Apparel,677,336
2,3,Adidas Running Shoes,Apparel,379,498
3,4,Allen Solly Formal Shirt,Apparel,1315,292
4,5,Nescafe Classic 200g,Grocery,1630,471
5,6,Prestige Pressure Cooker,Home,1933,421
6,7,Dell Inspiron 15,Electronics,1048,62
7,8,Puma Sports T-Shirt,Apparel,1393,358
8,9,Prestige Pressure Cooker,Home,2431,372
9,10,Dell Inspiron 15,Electronics,276,455


In [18]:
stores.head(10)

,store_id,store_name,store_city,store_region,opening_date
0,1,Reliance Digital  T Nagar,Mumbai,South,55:02.3
1,2,Croma  Forum Mall,Delhi,East,55:02.3
2,3,()Bazaar  Velachery,Chennai,East,55:02.3
3,4,Smart Bazaar  Whitefield,Bangalore,North,55:02.3
4,5,Reliance Trends  Banjara Hills,Mumbai,North,55:02.3
5,6,!More  Anna Nagar,Chennai,East,55:02.3
6,7,Reliance Smart  Andheri,Chennai,West,55:02.3
7,8,Spencer's  Salt Lake,Bangalore,South,55:02.3
8,9,DMart  Powai,Delhi,West,55:02.3
9,10,Reliance Fresh  Indiranagar,Delhi,North,55:02.3


In [24]:
sales_header["transaction_date"] = pd.to_datetime(sales_header["transaction_date"])

# Find sales date range
min_sales_date = sales_header["transaction_date"].min()
max_sales_date = sales_header["transaction_date"].max()

print("Sales start:", min_sales_date)
print("Sales end:", max_sales_date)

Sales start: 2025-08-09 08:55:02.261015
Sales end: 2026-02-04 08:55:02.261015


In [77]:
# Generate random opening dates 3–18 months before first sale
random_days = np.random.randint(90, 550, size=len(stores))

stores["opening_date"] = min_sales_date - pd.to_timedelta(random_days, unit="D")

# Convert nicely
stores["opening_date"] = pd.to_datetime(stores["opening_date"]).dt.date


In [89]:
sales_header["transaction_date"] = pd.to_datetime(
    sales_header["transaction_date"],
    errors="coerce"
)

mask = customers["last_purchase_date"].isna()

random_dates = pd.to_datetime(
    np.random.choice(
        pd.date_range(start_date, end_date),
        size=mask.sum()
    )
)

customers.loc[mask, "last_purchase_date"] = random_dates

customers["last_purchase_date"] = pd.to_datetime(
    customers["last_purchase_date"],
    errors="coerce"
)
sales_header["transaction_date"] = sales_header["transaction_date"].dt.strftime("%Y-%m-%d")

customers["last_purchase_date"] = customers["last_purchase_date"].dt.strftime("%Y-%m-%d")



In [78]:
stores.head(10)

,store_id,store_name,store_city,store_region,opening_date
0,1,Reliance Digital T Nagar,Mumbai,South,2024-12-15
1,2,Croma Forum Mall,Delhi,East,2024-03-11
2,3,Bazaar Velachery,Chennai,East,2024-05-21
3,4,Smart Bazaar Whitefield,Bangalore,North,2024-08-15
4,5,Reliance Trends Banjara Hills,Mumbai,North,2024-10-07
5,6,More Anna Nagar,Chennai,East,2025-04-18
6,7,Reliance Smart Andheri,Chennai,West,2024-06-13
7,8,Spencers Salt Lake,Bangalore,South,2024-03-06
8,9,DMart Powai,Delhi,West,2025-03-02
9,10,Reliance Fresh Indiranagar,Delhi,North,2024-07-26


In [63]:
import re

stores["store_name"] = (
    stores["store_name"]
    .astype(str)
    .str.replace(r"[^\w\s\-]", "", regex=True)   
    .str.replace(r"\s+", " ", regex=True)       
    .str.strip()
)

customers["last_purchase_date"] = pd.to_datetime(customers["last_purchase_date"], errors="coerce")
sales_header["transaction_date"] = pd.to_datetime(sales_header["transaction_date"], errors="coerce")



In [79]:
stores.head(10)

,store_id,store_name,store_city,store_region,opening_date
0,1,Reliance Digital T Nagar,Mumbai,South,2024-12-15
1,2,Croma Forum Mall,Delhi,East,2024-03-11
2,3,Bazaar Velachery,Chennai,East,2024-05-21
3,4,Smart Bazaar Whitefield,Bangalore,North,2024-08-15
4,5,Reliance Trends Banjara Hills,Mumbai,North,2024-10-07
5,6,More Anna Nagar,Chennai,East,2025-04-18
6,7,Reliance Smart Andheri,Chennai,West,2024-06-13
7,8,Spencers Salt Lake,Bangalore,South,2024-03-06
8,9,DMart Powai,Delhi,West,2025-03-02
9,10,Reliance Fresh Indiranagar,Delhi,North,2024-07-26


In [80]:
products.head(10)

,product_id,product_name,product_category,unit_price,current_stock_level
0,1,Adidas Running Shoes,Apparel,855,462
1,2,Puma Sports T-Shirt,Apparel,677,336
2,3,Adidas Running Shoes,Apparel,379,498
3,4,)Allen Solly Formal Shirt,Apparel,1315,292
4,5,Nescafe Classic 200g,Grocery,1630,471
5,6,Prestige Pressure Cooker,Home,1933,421
6,7,$Dell Inspiron 15,Electronics,1048,62
7,8,Puma Sports T-Shirt,Apparel,1393,358
8,9,Prestige Pressure Cooker,Home,2431,372
9,10,Dell Inspiron 15,Electronics,276,455


In [92]:
promotions.head(10)

,promotion_id,promotion_name,start_date,end_date,discount_percentage,applicable_category
0,1,Diwali Sale,2025-08-18 08:55:02.261015,2025-10-09 08:55:02.261015,0.10,Apparel
1,2,New Year Bonanza,2025-08-09 08:55:02.261015,2025-11-24 08:55:02.261015,0.05,Electronics
2,3,Weekend Flash Sale,2025-08-12 08:55:02.261015,2026-02-03 08:55:02.261015,0.05,ALL
3,4,Student Discount,2025-10-01 08:55:02.261015,2026-01-22 08:55:02.261015,0.10,Apparel
4,5,Summer Mega Offer,2025-09-30 08:55:02.261015,2025-11-16 08:55:02.261015,0.20,Grocery
5,6,Republic Day Deal,2025-08-22 08:55:02.261015,2025-12-15 08:55:02.261015,0.20,Grocery


In [67]:
loyalty_rules.head(10)

,rule_id,rule_name,points_per_unit_spend,min_spend_threshold,bonus_points
0,1,Standard Earning,1.0,0,0
1,2,Weekend Bonus,1.5,500,50
2,3,High Spender Bonus,2.0,1000,100


In [87]:
sales_header.head(10)

,transaction_id,customer_id,store_id,transaction_date,customer_phone,total_amount
0,TXN100000,1479,15,2025-09-11,9100001478,12625
1,TXN100001,3634,15,2025-08-31,9100003633,12340
2,TXN100002,1338,18,2025-12-22,9100001337,10806
3,TXN100003,4612,7,2025-09-04,9100004611,12618
4,TXN100004,7109,19,2025-12-01,9100007108,786
5,TXN100005,2841,7,2025-10-23,9100002840,5976
6,TXN100006,498,11,2025-08-16,9100000497,1522
7,TXN100007,7212,10,2025-08-23,9100007211,2560
8,TXN100008,1613,17,2025-09-27,9100001612,9273
9,TXN100009,3099,2,2025-11-02,9100003098,19359


In [41]:
sales_line_items.head(10)

,line_item_id,transaction_id,product_id,promotion_id,quantity,line_item_amount
0,1,TXN100000,122,1.0,3,6729
1,2,TXN100000,88,5.0,2,5896
2,3,TXN100001,179,5.0,3,2829
3,4,TXN100001,178,1.0,3,7611
4,5,TXN100001,92,1.0,4,1900
5,6,TXN100002,200,6.0,1,612
6,7,TXN100002,108,6.0,2,5298
7,8,TXN100002,57,1.0,2,4896
8,9,TXN100003,95,3.0,3,7698
9,10,TXN100003,10,6.0,2,552


In [90]:
customers.head(10)

,customer_id,customer_name,email,loyalty_status,total_loyalty_points,last_purchase_date,customer_phone,segment_name,customer_since_date
0,1,Santhosh,santhosh0@gmail.com,Bronze,2700,2025-01-10,9100000000,Premium High Value,01-10-2025 08:55
1,2,Divya,divya1@gmail.com,Gold,5015,2025-07-12,9100000001,Regular Shopper,25-12-2025 08:55
2,3,Divya,divya2@gmail.com,Gold,3430,2026-05-01,9100000002,Occasional Buyer,05-01-2026 08:55
3,4,Vijay,vijay3@gmail.com,Silver,4062,2025-11-12,9100000003,New Customer,11-12-2025 08:55
4,5,Meena,meena4@gmail.com,Silver,6082,2025-12-12,9100000004,Loyal Customer,12-12-2025 08:55
5,6,Priya,priya5@gmail.com,Gold,3605,2025-07-21,9100000005,Premium High Value,16-10-2025 08:55
6,7,Priya,priya6@gmail.com,Silver,6459,2025-10-04,9100000006,Regular Shopper,31-08-2025 08:55
7,8,Pooja,pooja7@gmail.com,Bronze,3987,2025-11-26,9100000007,Occasional Buyer,28-10-2025 08:55
8,9,Akash,akash8@gmail.com,Bronze,5425,2025-10-22,9100000008,New Customer,30-11-2025 08:55
9,10,Divya,divya9@gmail.com,Bronze,1084,2026-02-02,9100000009,Loyal Customer,02-02-2026 08:55


In [70]:
#RECOMPUTE total_amount
correct_totals = (
    sales_line_items
    .groupby("transaction_id")["line_item_amount"]
    .sum()
    .reset_index()
)

sales_header = sales_header.drop(columns="total_amount") \
                           .merge(correct_totals, on="transaction_id", how="left")

sales_header.rename(columns={"line_item_amount": "total_amount"}, inplace=True)


In [96]:
#Loyalty_points
master["transaction_date"] = pd.to_datetime(
    master["transaction_date"],
    errors="coerce"
)
behavior = master.groupby("customer_id").agg({
    "total_amount": "sum",
    "transaction_id": "count",
    "transaction_date": lambda x: (master["transaction_date"].max() - x.max()).days,
    "product_category": "nunique",
    "promotion_id": lambda x: x.notna().mean()
}).reset_index()

behavior.columns = [
    "customer_id", "total_spend", "frequency",
    "recency_days", "category_diversity", "promo_ratio"
]
def normalize(series):
    return 100 * (series - series.min()) / (series.max() - series.min())

behavior["spend_score"] = normalize(behavior["total_spend"])
behavior["freq_score"] = normalize(behavior["frequency"])
behavior["recency_score"] = 100 - normalize(behavior["recency_days"])
behavior["diversity_score"] = normalize(behavior["category_diversity"])
behavior["profit_score"] = 100 - normalize(behavior["promo_ratio"])

recent = master[master["transaction_date"] >= master["transaction_date"].max() - pd.Timedelta(days=30)]
old = master[master["transaction_date"] < master["transaction_date"].max() - pd.Timedelta(days=30)]

recent_spend = recent.groupby("customer_id")["total_amount"].sum()
old_spend = old.groupby("customer_id")["total_amount"].sum()

behavior["growth"] = behavior["customer_id"].map(
    (recent_spend - old_spend).fillna(0)
)

behavior["growth_score"] = normalize(behavior["growth"])

behavior["loyalty_score"] = (
    0.30 * behavior["spend_score"] +
    0.25 * behavior["freq_score"] +
    0.20 * behavior["recency_score"] +
    0.10 * behavior["diversity_score"] +
    0.10 * behavior["profit_score"] +
    0.05 * behavior["growth_score"]
)

behavior["loyalty_score"] = behavior["loyalty_score"].round(2)



In [103]:
#loyalty status
behavior["loyalty_status"] = pd.qcut(
    behavior["loyalty_score"],
    q=4,
    labels=["Bronze", "Silver", "Gold", "Platinum"]
)
def assign_tier(score):
    if score >= 80:
        return "Platinum"
    elif score >= 60:
        return "Gold"
    elif score >= 40:
        return "Silver"
    else:
        return "Bronze"

behavior["loyalty_status"] = behavior["loyalty_score"].apply(assign_tier)

behavior[["customer_id", "loyalty_score", "loyalty_status"]].head()


,customer_id,loyalty_score,loyalty_status
0,1,51.72,Silver
1,2,68.63,Gold
2,3,47.05,Silver
3,4,52.51,Silver
4,5,35.97,Bronze


In [53]:
#RFM Calculation
today = sales_header["transaction_date"].max()

rfm = (
    sales_header.groupby("customer_id")
    .agg({
        "transaction_date": lambda x: (today - x.max()).days,
        "transaction_id": "count",
        "total_amount": "sum"
    })
    .reset_index()
)

rfm.columns = ["customer_id", "recency", "frequency", "monetary"]


In [54]:
rfm.head()

,customer_id,recency,frequency,monetary
0,1,6,9,74108
1,2,9,15,145980
2,3,2,8,62289
3,4,1,10,94414
4,5,18,4,28632


In [55]:
# customer segmentation based on the spending
threshold = rfm["monetary"].quantile(0.9)

rfm["segment"] = np.where(
    rfm["monetary"] >= threshold,
    "High Spender",
    "Regular"
)

In [56]:
rfm.loc[rfm["recency"] > 30, "segment"] = "At Risk"


In [113]:
# customer lifetime value calculation

def normalize(series):
    return (series - series.min()) / (series.max() - series.min())

rfm["clv_score"] = (
    0.5 * normalize(rfm["monetary"]) +
    0.3 * normalize(rfm["frequency"]) +
    0.2 * (1 - normalize(rfm["recency"]))
)

rfm["clv_score"] = (rfm["clv_score"] * 100).round(2)

rfm.head()

,customer_id,recency,frequency,monetary,segment,clv_score
0,1,6,9,74108,Regular,50.03
1,2,9,15,145980,High Spender,76.91
2,3,2,8,62289,Regular,45.98
3,4,1,10,94414,Regular,57.19
4,5,18,4,28632,Regular,29.54
